# Gold diagnostics — optimized

Per Gold table, a single aggregation computes:
- total row count
- required-column null counts
- exact duplicate row count
- table-specific invalid-value counts

Separate operations are retained only for checks that genuinely require
grouping, joins, or temporal ordering.


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.functions import broadcast
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    DoubleType,
)
import json

CATALOG = "data_lakehouse_databricks"
GOLD_SCHEMA = "gold"
DIAGNOSTIC_TABLE = (
    f"{CATALOG}.{GOLD_SCHEMA}.diagnostic_gold"
)

TABLE_CONFIG = {
    "gold_dim_customers": {
        "keys": ["customer_id"],
        "secondary_unique_keys": [
            ["customer_key"]
        ],
        "required_columns": [
            "customer_id",
            "customer_key",
            "firstname",
            "lastname",
            "marital_status",
            "gender",
            "country",
            "birth_date",
            "creation_date",
        ],
        "invalid_conditions": {
            "future_birth_date": (
                F.col("birth_date")
                > F.current_date()
            ),
        },
    },

    "gold_dim_product": {
        "keys": ["product_id"],
        "secondary_unique_keys": [
            ["product_key", "start_date"]
        ],
        "required_columns": [
            "product_id",
            "product_key",
            "product_name",
            "cost",
            "product_line",
            "start_date",
            "category_id",
            "category",
            "subcategory",
            "maintenance",
        ],
        "invalid_conditions": {
            "negative_cost": (
                F.col("cost") < 0
            ),
            "invalid_date_range": (
                F.col("end_date").isNotNull()
                & (
                    F.col("end_date")
                    < F.col("start_date")
                )
            ),
        },
    },

    "gold_fact_sales": {
        "keys": [
            "order_number",
            "product_key",
        ],
        "secondary_unique_keys": [],
        "required_columns": [
            "order_number",
            "product_key",
            "customer_id",
            "order_date",
            "ship_date",
            "due_date",
            "sales_amount",
            "quantity",
            "price",
        ],
        "invalid_conditions": {
            "negative_sales_amount": (
                F.col("sales_amount") < 0
            ),
            "non_positive_quantity": (
                F.col("quantity") <= 0
            ),
            "negative_price": (
                F.col("price") < 0
            ),
            "ship_before_order": (
                F.col("ship_date").isNotNull()
                & F.col("order_date").isNotNull()
                & (
                    F.col("ship_date")
                    < F.col("order_date")
                )
            ),
            "due_before_order": (
                F.col("due_date").isNotNull()
                & F.col("order_date").isNotNull()
                & (
                    F.col("due_date")
                    < F.col("order_date")
                )
            ),
        },
    },
}

RESULT_SCHEMA = StructType([
    StructField(
        "table_name",
        StringType(),
        False,
    ),
    StructField(
        "check_name",
        StringType(),
        False,
    ),
    StructField(
        "check_type",
        StringType(),
        False,
    ),
    StructField(
        "columns_checked",
        StringType(),
        True,
    ),
    StructField(
        "total_rows",
        LongType(),
        False,
    ),
    StructField(
        "failed_rows",
        LongType(),
        False,
    ),
    StructField(
        "pass_percentage",
        DoubleType(),
        False,
    ),
    StructField(
        "status",
        StringType(),
        False,
    ),
    StructField(
        "details",
        StringType(),
        True,
    ),
])


def make_result(
    table_name,
    check_name,
    check_type,
    total_rows,
    failed_rows,
    columns_checked=None,
    details=None,
    warn_only=False,
):
    total_rows = int(total_rows)
    failed_rows = int(failed_rows)

    if total_rows == 0:
        pass_percentage = (
            100.0
            if failed_rows == 0
            else 0.0
        )
    else:
        pass_percentage = round(
            100.0
            * (
                total_rows
                - failed_rows
            )
            / total_rows,
            3,
        )

    if failed_rows == 0:
        status = "PASS"
    elif warn_only:
        status = "WARN"
    else:
        status = "FAIL"

    return {
        "table_name": table_name,
        "check_name": check_name,
        "check_type": check_type,
        "columns_checked": (
            json.dumps(
                columns_checked
            )
            if columns_checked
            is not None
            else None
        ),
        "total_rows": total_rows,
        "failed_rows": failed_rows,
        "pass_percentage": (
            pass_percentage
        ),
        "status": status,
        "details": details,
    }


def duplicate_rows_for_keys(
    df,
    keys,
):
    grouped = (
        df
        .groupBy(*keys)
        .count()
        .filter(
            F.col("count") > 1
        )
    )

    row = (
        grouped
        .agg(
            F.coalesce(
                F.sum(
                    F.col("count")
                    - F.lit(1)
                ),
                F.lit(0),
            )
            .cast("long")
            .alias(
                "duplicate_rows"
            )
        )
        .first()
    )

    return int(
        row[
            "duplicate_rows"
        ] or 0
    )


In [ ]:
results = []
loaded_tables = {}
total_rows_by_table = {}
required_nulls_by_table = {}

for table_name, config in TABLE_CONFIG.items():
    full_name = (
        f"{CATALOG}.{GOLD_SCHEMA}.{table_name}"
    )

    if not spark.catalog.tableExists(
        full_name
    ):
        results.append(
            make_result(
                table_name=table_name,
                check_name="table_exists",
                check_type="existence",
                total_rows=1,
                failed_rows=1,
                details=(
                    f"Table not found: {full_name}"
                ),
            )
        )
        continue

    df = spark.table(full_name)
    loaded_tables[table_name] = df
    columns = df.columns

    missing_columns = sorted(
        set(
            config[
                "required_columns"
            ]
        )
        - set(columns)
    )

    if missing_columns:
        raise RuntimeError(
            f"{full_name} is missing "
            f"configured columns: "
            f"{missing_columns}"
        )

    # One aggregation for:
    # row count + null checks + exact duplicates + invalid-value checks.
    expressions = [
        F.count(F.lit(1))
        .cast("long")
        .alias("_row_count"),

        F.countDistinct(
            F.struct(
                *[
                    F.col(c)
                    for c in columns
                ]
            )
        )
        .cast("long")
        .alias("_distinct_rows"),
    ]

    required_columns = (
        config[
            "required_columns"
        ]
    )

    for i, column_name in enumerate(
        required_columns
    ):
        expressions.append(
            F.sum(
                F.when(
                    F.col(
                        column_name
                    ).isNull(),
                    F.lit(1),
                ).otherwise(
                    F.lit(0)
                )
            )
            .cast("long")
            .alias(
                f"_null_{i}"
            )
        )

    invalid_items = list(
        config.get(
            "invalid_conditions",
            {},
        ).items()
    )

    for i, (
        _,
        condition,
    ) in enumerate(
        invalid_items
    ):
        expressions.append(
            F.sum(
                F.when(
                    condition,
                    F.lit(1),
                ).otherwise(
                    F.lit(0)
                )
            )
            .cast("long")
            .alias(
                f"_invalid_{i}"
            )
        )

    metrics = df.agg(
        *expressions
    ).first()

    total_rows = int(
        metrics[
            "_row_count"
        ] or 0
    )
    total_rows_by_table[
        table_name
    ] = total_rows

    distinct_rows = int(
        metrics[
            "_distinct_rows"
        ] or 0
    )
    exact_duplicate_rows = (
        total_rows
        - distinct_rows
    )

    null_counts = {}

    for i, column_name in enumerate(
        required_columns
    ):
        null_counts[
            column_name
        ] = int(
            metrics[
                f"_null_{i}"
            ] or 0
        )

    required_nulls_by_table[
        table_name
    ] = null_counts

    results.append(
        make_result(
            table_name=table_name,
            check_name="table_exists",
            check_type="existence",
            total_rows=1,
            failed_rows=0,
            details=(
                f"Table exists with "
                f"{total_rows} rows."
            ),
        )
    )

    results.append(
        make_result(
            table_name=table_name,
            check_name="table_not_empty",
            check_type="row_count",
            total_rows=max(
                total_rows,
                1,
            ),
            failed_rows=(
                1
                if total_rows == 0
                else 0
            ),
            details=(
                f"Row count: "
                f"{total_rows}"
            ),
        )
    )

    results.append(
        make_result(
            table_name=table_name,
            check_name=(
                "exact_duplicate_rows"
            ),
            check_type="duplication",
            total_rows=total_rows,
            failed_rows=(
                exact_duplicate_rows
            ),
            columns_checked=columns,
        )
    )

    for column_name in required_columns:
        results.append(
            make_result(
                table_name=table_name,
                check_name=(
                    "required_not_null:"
                    f"{column_name}"
                ),
                check_type="nullability",
                total_rows=total_rows,
                failed_rows=(
                    null_counts[
                        column_name
                    ]
                ),
                columns_checked=[
                    column_name
                ],
            )
        )

    for i, (
        check_name,
        _,
    ) in enumerate(
        invalid_items
    ):
        failed_rows = int(
            metrics[
                f"_invalid_{i}"
            ] or 0
        )

        results.append(
            make_result(
                table_name=table_name,
                check_name=check_name,
                check_type=(
                    "business_rule"
                ),
                total_rows=total_rows,
                failed_rows=(
                    failed_rows
                ),
            )
        )

    # These require a real grouping/shuffle and therefore remain separate.
    all_key_sets = [
        config["keys"],
        *config.get(
            "secondary_unique_keys",
            [],
        ),
    ]

    for keys in all_key_sets:
        duplicate_rows = (
            duplicate_rows_for_keys(
                df,
                keys,
            )
        )

        results.append(
            make_result(
                table_name=table_name,
                check_name=(
                    "unique_key:"
                    + ",".join(keys)
                ),
                check_type="uniqueness",
                total_rows=total_rows,
                failed_rows=(
                    duplicate_rows
                ),
                columns_checked=keys,
            )
        )


In [ ]:
# Referential integrity: fact -> customer dimension.
if {
    "gold_fact_sales",
    "gold_dim_customers",
}.issubset(loaded_tables):
    fact = loaded_tables[
        "gold_fact_sales"
    ]

    customer_keys = (
        loaded_tables[
            "gold_dim_customers"
        ]
        .select("customer_id")
        .distinct()
    )

    unmatched_customers = (
        fact
        .select("customer_id")
        .join(
            broadcast(
                customer_keys
            ),
            "customer_id",
            "left_anti",
        )
        .count()
    )

    results.append(
        make_result(
            table_name=(
                "gold_fact_sales"
            ),
            check_name=(
                "customer_referential_integrity"
            ),
            check_type=(
                "referential_integrity"
            ),
            total_rows=(
                total_rows_by_table[
                    "gold_fact_sales"
                ]
            ),
            failed_rows=(
                unmatched_customers
            ),
            columns_checked=[
                "customer_id"
            ],
        )
    )

# Referential + temporal product integrity.
if {
    "gold_fact_sales",
    "gold_dim_product",
}.issubset(loaded_tables):
    fact = loaded_tables[
        "gold_fact_sales"
    ]
    products = loaded_tables[
        "gold_dim_product"
    ]

    product_keys = (
        products
        .select("product_key")
        .distinct()
    )

    unmatched_products = (
        fact
        .select("product_key")
        .join(
            broadcast(
                product_keys
            ),
            "product_key",
            "left_anti",
        )
        .count()
    )

    results.append(
        make_result(
            table_name=(
                "gold_fact_sales"
            ),
            check_name=(
                "product_referential_integrity"
            ),
            check_type=(
                "referential_integrity"
            ),
            total_rows=(
                total_rows_by_table[
                    "gold_fact_sales"
                ]
            ),
            failed_rows=(
                unmatched_products
            ),
            columns_checked=[
                "product_key"
            ],
        )
    )

    # Historical product version active on the fact's order date.
    f = fact.alias("f")
    p = (
        products
        .select(
            "product_key",
            "start_date",
            "end_date",
        )
        .alias("p")
    )

    historical_condition = (
        (
            F.col(
                "f.product_key"
            )
            == F.col(
                "p.product_key"
            )
        )
        & (
            F.col(
                "f.order_date"
            )
            >= F.col(
                "p.start_date"
            )
        )
        & (
            F.col(
                "p.end_date"
            ).isNull()
            | (
                F.col(
                    "f.order_date"
                )
                <= F.col(
                    "p.end_date"
                )
            )
        )
    )

    historical_misses = (
        f
        .filter(
            F.col(
                "f.order_date"
            ).isNotNull()
        )
        .join(
            broadcast(p),
            historical_condition,
            "left_anti",
        )
        .count()
    )

    order_date_nulls = (
        required_nulls_by_table[
            "gold_fact_sales"
        ]["order_date"]
    )

    dated_fact_rows = (
        total_rows_by_table[
            "gold_fact_sales"
        ]
        - order_date_nulls
    )

    results.append(
        make_result(
            table_name=(
                "gold_fact_sales"
            ),
            check_name=(
                "historical_product_version_match"
            ),
            check_type=(
                "temporal_integrity"
            ),
            total_rows=(
                dated_fact_rows
            ),
            failed_rows=(
                historical_misses
            ),
            columns_checked=[
                "product_key",
                "order_date",
            ],
        )
    )

    # Detect overlapping validity periods per product_key.
    product_window = (
        Window
        .partitionBy(
            "product_key"
        )
        .orderBy(
            F.col(
                "start_date"
            ).asc_nulls_last()
        )
    )

    product_periods = (
        products
        .select(
            "product_key",
            "start_date",
            "end_date",
        )
        .withColumn(
            "_previous_start",
            F.lag(
                "start_date"
            ).over(
                product_window
            ),
        )
        .withColumn(
            "_previous_end",
            F.lag(
                "end_date"
            ).over(
                product_window
            ),
        )
    )

    overlap_rows = (
        product_periods
        .filter(
            F.col(
                "_previous_start"
            ).isNotNull()
            & (
                F.col(
                    "_previous_end"
                ).isNull()
                | (
                    F.col(
                        "start_date"
                    )
                    <= F.col(
                        "_previous_end"
                    )
                )
            )
        )
        .count()
    )

    results.append(
        make_result(
            table_name=(
                "gold_dim_product"
            ),
            check_name=(
                "product_validity_overlap"
            ),
            check_type=(
                "temporal_integrity"
            ),
            total_rows=(
                total_rows_by_table[
                    "gold_dim_product"
                ]
            ),
            failed_rows=(
                overlap_rows
            ),
            columns_checked=[
                "product_key",
                "start_date",
                "end_date",
            ],
        )
    )


In [ ]:
results_df = (
    spark.createDataFrame(
        results,
        schema=RESULT_SCHEMA,
    )
    .orderBy(
        "table_name",
        "check_type",
        "check_name",
    )
)

(
    results_df.write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true",
    )
    .saveAsTable(
        DIAGNOSTIC_TABLE
    )
)

failed_checks = sum(
    row["status"] == "FAIL"
    for row in results
)

print(
    f"Saved Gold diagnostics: "
    f"{DIAGNOSTIC_TABLE}. "
    f"Checks={len(results)}, "
    f"failures={failed_checks}"
)

dbutils.notebook.exit(
    f"OK|{DIAGNOSTIC_TABLE}|"
    f"checks={len(results)}|"
    f"failures={failed_checks}"
)
